# Day 040 — Exercise 1: summarize_column

**What you'll build:** `summarize_column(col_name, stats, model) -> str` — take a column name and its `distribution_summary` stats dict, build a prompt, call Ollama, and return a 1-2 sentence description in plain English.

**Why it matters:** Numbers alone don't communicate. A stats dict tells you mean=298.69 and std=453.04 — an LLM turns that into 'Revenue varies widely, from as low as $100 to a peak of $1,050, with most orders in the $125-$450 range.' That sentence is what a stakeholder reads.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import ollama
import pandas as pd
import io


def distribution_summary(df, col):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count': int(s.count()), 'mean': round(float(s.mean()), 4),
            'std': round(float(s.std()), 4), 'min': float(s.min()),
            'q25': float(s.quantile(0.25)), 'median': float(s.quantile(0.50)),
            'q75': float(s.quantile(0.75)), 'max': float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count': int(s.count()), 'unique': int(s.nunique()),
        'top': str(counts.index[0]) if len(counts) else None,
        'top_freq': int(counts.iloc[0]) if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

## Your Implementation

In [ ]:
def summarize_column(col_name: str, stats: dict,
                     model: str = 'llama3.2') -> str:
    """
    Describe one column in plain English using an LLM.

    Args:
        col_name — column name to describe (included in prompt)
        stats    — dict from distribution_summary (numeric or categorical)
        model    — Ollama model to use
    Returns:
        str      — 1-2 sentence plain-English description
    """
    # TODO: build a prompt string that includes col_name and
    #       json.dumps(stats, indent=2)
    # TODO: resp = ollama.chat(model=model,
    #             messages=[{'role': 'user', 'content': prompt}])
    # TODO: return resp['message']['content'].strip()
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0
    _result = None

    # Check 1: function defined
    try:
        assert 'summarize_column' in globals()
        passed += 1; print('\u2705 Check 1: summarize_column is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string (makes a real Ollama call)
    try:
        _stats = distribution_summary(SALES_DF, 'revenue')
        _result = summarize_column('revenue', _stats)
        assert isinstance(_result, str), \
            f'expected str, got {type(_result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: response is non-empty
    try:
        assert len(_result.strip()) > 0, 'response is empty'
        passed += 1; print('\u2705 Check 3: response is non-empty')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: response is a meaningful length (> 30 chars)
    try:
        _n = len(_result.strip())
        assert _n > 30, \
            f'response too short ({_n} chars) — expected a meaningful sentence'
        passed += 1; print(f'\u2705 Check 4: response is {_n} chars')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: also works for a categorical column
    try:
        _cat_stats = distribution_summary(SALES_DF, 'product')
        _cat_result = summarize_column('product', _cat_stats)
        assert isinstance(_cat_result, str) and len(_cat_result.strip()) > 0
        passed += 1; print('\u2705 Check 5: works for categorical column (product)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import json
import ollama

def summarize_column(col_name: str, stats: dict,
                     model: str = 'llama3.2') -> str:
    prompt = (
        f"You are a concise data analyst. Describe the column '{col_name}' "
        f"in 1-2 clear sentences for a non-technical reader.\n\n"
        f"Statistics:\n{json.dumps(stats, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()
```

</details>